In [4]:
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

COUGHVID_CLEAN = Path("../data/coughvid/coughvid_clean.csv")
CAMBRIDGE_CLEAN = Path("../data/cambridge/cambridge_clean.csv")
OUTPUT_DIR = Path("../data")

ModuleNotFoundError: No module named 'librosa'

In [ ]:
cv = pd.read_csv(COUGHVID_CLEAN)
cam = pd.read_csv(CAMBRIDGE_CLEAN)

print(f"COUGHVID rows:   {len(cv)}")
print(f"Cambridge rows:  {len(cam)}")

In [ ]:
def load_audio(path, sr=22050, max_duration=5.0):
    try:
        y, _ = librosa.load(path, sr=sr, mono=True, duration=max_duration)
        # trim leading/trailing silence
        y, _ = librosa.effects.trim(y, top_db=20)
        # skip clips that are too short after trimming
        if len(y) < sr * 0.3:
            return None
        # peak normalise
        if np.abs(y).max() > 0:
            y = y / np.abs(y).max()
        return y
    except Exception:
        return None

In [ ]:
def extract_features(path, sr=22050, n_mfcc=13):
    y = load_audio(path, sr=sr)
    if y is None:
        return None

    feats = []

    def agg(arr):
        # arr shape: (n_features, time_frames)
        return np.concatenate([
            arr.mean(axis=1),
            arr.std(axis=1),
            arr.max(axis=1)
        ])

    # --- MFCCs + delta + delta-delta ---
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_d  = librosa.feature.delta(mfcc)
    mfcc_d2 = librosa.feature.delta(mfcc, order=2)
    for arr in [mfcc, mfcc_d, mfcc_d2]:
        feats.append(agg(arr))          # 3 × (13×3) = 117 values

    # --- Spectral features ---
    sc  = librosa.feature.spectral_centroid(y=y, sr=sr)     # (1, T)
    sb  = librosa.feature.spectral_bandwidth(y=y, sr=sr)    # (1, T)
    sro = librosa.feature.spectral_rolloff(y=y, sr=sr)      # (1, T)
    sco = librosa.feature.spectral_contrast(y=y, sr=sr)     # (7, T)
    for arr in [sc, sb, sro]:
        feats.append(agg(arr))          # 3 × (1×3) = 9 values
    feats.append(agg(sco))              # 7×3 = 21 values

    # --- Temporal features ---
    zcr = librosa.feature.zero_crossing_rate(y)             # (1, T)
    rms = librosa.feature.rms(y=y)                          # (1, T)
    for arr in [zcr, rms]:
        feats.append(agg(arr))          # 2 × (1×3) = 6 values

    # --- Chroma ---
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)        # (12, T)
    feats.append(agg(chroma))           # 12×3 = 36 values

    # --- Duration ---
    feats.append(np.array([len(y) / sr]))   # 1 value

    return np.concatenate(feats).astype(np.float32)

In [ ]:
# Test on first COUGHVID file
test_path = cv['audio_path'].iloc[0]
print(f"Testing on: {test_path}")

vec = extract_features(test_path)
if vec is not None:
    print(f"Feature vector length: {len(vec)}")
    print(f"First 10 values: {vec[:10].round(3)}")
    print(f"Any NaN: {np.isnan(vec).any()}")
else:
    print("Returned None — check audio path")